In [ ]:
# Instalar el ecosistema core de LangChain y la integración con Groq
%pip install -qU langchain langchain-groq langchain-community

# Instalar herramientas de búsqueda y persistencia asíncrona para SQLite
%pip install -qU tavily-python langchain-tavily aiosqlite

# Instalar el motor de LangGraph y su checkpointer oficial
%pip install -qU langgraph langgraph-checkpoint-sqlite

In [ ]:
import operator
import sqlite3
from typing import Annotated, List, TypedDict

from langchain_core.messages import (
    AIMessage,
    AnyMessage,
    ChatMessage,
    HumanMessage,
    SystemMessage,
)
from langgraph.checkpoint.sqlite import SqliteSaver  # Asegurá tener langgraph-checkpoint-sqlite instalado
from langgraph.graph import END, StateGraph

# Configurar la conexión persistente a la base de datos local
# check_same_thread=False es clave en Windows para que Jupyter no bloquee los hilos de SQLite
conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
memory = SqliteSaver(conn)

In [ ]:
import os
from dotenv import load_dotenv

# Cargar las variables de entorno desde tu archivo .env
load_dotenv()

# Configurar las llaves API correctas para tu arquitectura con Groq
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

In [ ]:
import operator
from typing import Annotated, List, TypedDict


class AgentState(TypedDict):
    task: str
    plan: str
    draft: str
    critique: str
    # Usamos Annotated y operator.add para que LangGraph sume/acumule los elementos 
    # en la lista en lugar de sobreescribir todo el campo.
    content: Annotated[List[str], operator.add]
    revision_number: int
    max_revisions: int

In [ ]:
from langchain_groq import ChatGroq

# Instanciamos Llama 3.3 de 70B, ideal para el patrón de planificación y crítica
model = ChatGroq(
    model="llama-3.3-70b-versatile", 
    temperature=0
)

In [ ]:
PLAN_PROMPT = """Eres un escritor especialista con la tarea de crear un esquema de alto nivel para una redacción. 
Escribe este esquema para el tema proporcionado por el usuario. 

Presenta un plan estructurado de la redacción junto con las notas o instrucciones relevantes para cada una de las secciones.
Responde directamente con el esquema utilizando formato Markdown claro."""

In [ ]:
WRITER_PROMPT = """Eres un asistente de redacción con la tarea de escribir excelentes redacciones de 5 párrafos. 
Genera la mejor redacción posible para la solicitud del usuario y el esquema inicial. 

Si el usuario o el nodo de crítica proporcionan observaciones, responde con una versión revisada y mejorada de tus intentos anteriores. 

Utiliza toda la información recopilada a continuación para redactar o mejorar el texto:
------
{content}"""

In [ ]:
REFLECTION_PROMPT = """Eres un profesor universitario experto encargado de calificar y corregir una redacción presentada. 
Genera una crítica constructiva y recomendaciones específicas para la entrega del usuario. 

Proporciona sugerencias detalladas, incluyendo observaciones sobre la extensión, profundidad de los argumentos, estilo, claridad y coherencia.
Sé estricto y directo, enfocándote en los puntos débiles que deben ser corregidos en la próxima revisión."""

In [ ]:
RESEARCH_PLAN_PROMPT = """Eres un investigador experto encargado de proporcionar información clave para la redacción. 
Tu tarea es generar una lista de consultas de investigación en texto plano que sirvan para buscar en la web y recopilar datos relevantes. 

Genera como máximo 3 consultas de búsqueda. 
Devuelve únicamente las consultas, una por línea, sin numeración, sin viñetas y sin introducciones."""

In [ ]:
RESEARCH_CRITIQUE_PROMPT = """Eres un investigador experto encargado de buscar información en la web para resolver las revisiones solicitadas. 
Basándote en la crítica proporcionada, genera una lista de consultas de investigación que recopilen los datos faltantes o corrijan las imprecisiones. 

Genera como máximo 3 consultas de búsqueda en texto plano. 
Devuelve únicamente las consultas, una por línea, sin numeración, sin viñetas y sin textos de introducción."""

In [ ]:
from typing import List
from pydantic import BaseModel, Field


class Queries(BaseModel):
    queries: List[str] = Field(
        description="Lista de hasta 3 consultas de búsqueda detalladas y específicas para la web."
    )

In [ ]:
import os
from tavily import TavilyClient

# Instanciar el cliente oficial de Tavily utilizando la API Key cargada previamente
tavily = TavilyClient(api_key=TAVILY_API_KEY)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage


def plan_node(state: AgentState):
    # Construimos el contexto para el planificador usando tus prompts optimizados
    messages = [
        SystemMessage(content=PLAN_PROMPT), 
        HumanMessage(content=state['task'])
    ]
    
    # Invocamos a Llama 3.3 a través de Groq
    response = model.invoke(messages)
    
    # Retornamos el plan en formato string (Markdown) para actualizar el AgentState
    return {"plan": response.content}

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage


def research_plan_node(state: AgentState):
    # 1. Obtener de forma estructurada las consultas usando Llama 3.3 y Pydantic
    structured_model = model.with_structured_output(Queries)
    queries = structured_model.invoke([
        SystemMessage(content=RESEARCH_PLAN_PROMPT),
        HumanMessage(content=state['task'])
    ])
    
    # 2. Creamos una lista NUEVA y vacía para guardar únicamente los resultados de este paso
    new_content = []
    
    # 3. Ejecutar las búsquedas en Tavily
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            new_content.append(r['content'])
            
    # 4. Retornamos SOLO lo nuevo. 
    # El reductor (operator.add) de tu AgentState lo va a fusionar automáticamente sin duplicar nada.
    return {"content": new_content}

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage


def generation_node(state: AgentState):
    # 1. Unimos todos los hallazgos de investigación acumulados en el estado
    content = "\n\n".join(state.get('content') or [])
    
    # 2. Estructuramos el mensaje del usuario con la tarea y el plan de Markdown
    user_message = HumanMessage(
        content=f"Tarea original: {state['task']}\n\nAquí está el esquema/plan a seguir:\n\n{state['plan']}"
    )
    
    # 3. Formateamos el prompt del sistema inyectándole la base de conocimientos extraída
    messages = [
        SystemMessage(
            content=WRITER_PROMPT.format(content=content)
        ),
        user_message
    ]
    
    # 4. Invocamos a Llama 3.3 para redactar el borrador de 5 párrafos
    response = model.invoke(messages)
    
    # 5. Retornamos el borrador (draft). 
    # Mantenemos el revision_number actual (si no existe, lo inicializamos en 1).
    # La suma (+1) la dejamos exclusivamente para el nodo que hace el "re-work".
    return {
        "draft": response.content, 
        "revision_number": state.get("revision_number") or 1
    }

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage


def reflection_node(state: AgentState):
    # 1. Construimos el contexto de evaluación pasándole el borrador al profesor
    messages = [
        SystemMessage(content=REFLECTION_PROMPT), 
        HumanMessage(content=state['draft'])
    ]
    
    # 2. Invocamos a Llama 3.3 para generar la crítica detallada
    response = model.invoke(messages)
    
    # 3. Extraemos el contador actual (por defecto 1 si no existiera)
    current_revision = state.get("revision_number") or 1
    
    # 4. Retornamos la crítica y sumamos +1 al contador de revisiones.
    # Esto asegura que si el grafo decide volver al escritor, este sepa que va por un nuevo intento.
    return {
        "critique": response.content,
        "revision_number": current_revision + 1
    }

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage


def research_critique_node(state: AgentState):
    # 1. Forzar la salida estructurada de consultas basadas en la crítica del profesor
    structured_model = model.with_structured_output(Queries)
    queries = structured_model.invoke([
        SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
        HumanMessage(content=state['critique'])
    ])
    
    # 2. Inicializar una lista NUEVA y vacía para capturar únicamente los datos de esta revisión
    new_content = []
    
    # 3. Iterar las consultas en Tavily para traer información de respaldo fresca
    for q in queries.queries:
        response = tavily.search(query=q, max_results=2)
        for r in response['results']:
            new_content.append(r['content'])
            
    # 4. Retornamos SOLO el nuevo contenido extraído.
    # LangGraph lo sumará automáticamente al historial sin duplicar lo que ya teníamos.
    return {"content": new_content}

In [ ]:
def should_continue(state: AgentState):
    current_revision = state.get("revision_number") or 1
    max_allowed = state.get("max_revisions") or 2
    
    if current_revision > max_allowed:
        return END
        
    # Si le quedan intentos, mandamos el texto al profesor ("reflect")
    return "reflect"

In [ ]:
from langgraph.graph import StateGraph, START, END

# 1. Reiniciamos el objeto para limpiar cualquier edge o nodo viejo en memoria
builder = StateGraph(AgentState)

# 2. Registramos los nodos
builder.add_node("planner", plan_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("generation", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_critique", research_critique_node)

# 3. Conexiones lineales
builder.add_edge(START, "planner")
builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generation")
builder.add_edge("generation", "reflect")

# 4. Conexión condicional (Esta es la que te tiraba error por duplicarse)
builder.add_conditional_edges(
    "reflect",
    should_continue,
    {
        "research_critique": "research_critique",
        END: END
    }
)

# 5. Cierre del bucle
builder.add_edge("research_critique", "generation")

# 6. Compilación
graph = builder.compile()
print("¡Grafo compilado con éxito y libre de duplicados!")

In [ ]:
from langgraph.graph import START

# Conectamos el punto de partida virtual (START) directamente con el nodo planificador
builder.add_edge(START, "planner")

In [ ]:
from langgraph.graph import StateGraph, START, END

# 1. Constructor de cero
builder = StateGraph(AgentState)

# 2. Registramos nodos (Usamos "generate" como en la captura)
builder.add_node("planner", plan_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("generate", generation_node)  
builder.add_node("reflect", reflection_node)
builder.add_node("research_critique", research_critique_node)

# 3. Camino lineal de entrada
builder.add_edge(START, "planner")
builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generate")

# 4. El condicional saliendo de "generate" (Como las líneas punteadas del video)
builder.add_conditional_edges(
    "generate",        # <--- CAMBIADO: El origen ahora es el escritor
    should_continue,
    {
        "reflect": "reflect",  # Si continúa, va a examen con el profesor
        END: END
    }
)

# 5. El retorno del ciclo según la captura de Alura
builder.add_edge("reflect", "research_critique")       # Del profesor va a investigar la crítica
builder.add_edge("research_critique", "generate")      # De la investigación vuelve a escribir

# 6. Compilación
graph = builder.compile(checkpointer=memory)
print("🎯 ¡Grafo re-estructurado con el modelo exacto de Alura!")

In [ ]:
from langgraph.graph import StateGraph, START, END

# 1. Reiniciamos el objeto por completo para limpiar la memoria interna de LangGraph
builder = StateGraph(AgentState)

# 2. Registramos los nodos (aquí LangGraph cargará las últimas versiones de tus funciones)
builder.add_node("planner", plan_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("generation", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_critique", research_critique_node)

# 3. Conexiones lineales iniciales
builder.add_edge(START, "planner")
builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generation")
builder.add_edge("generation", "reflect")

# 4. Conexión condicional (Ahora sí se registrará de forma limpia)
builder.add_conditional_edges(
    "reflect",
    should_continue,
    {
        "research_critique": "research_critique",
        END: END
    }
)

# 5. Cierre del bucle de retroalimentación
builder.add_edge("research_critique", "generation")

# 6. Compilación del ejecutable final
graph = builder.compile()
print("🎉 ¡Grafo re-compilado con éxito con tus últimas modificaciones!")

In [ ]:
# 1. Conexiones lineales del flujo inicial
builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generation")  # Corregido: "generation"

# 2. Cierre del bucle de optimización
builder.add_edge("research_critique", "generation")  # Corregido: "generation"

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# 1. Instanciamos el checkpointer en memoria
memory = MemorySaver()

# 2. Compilamos el grafo vinculándole la persistencia
graph = builder.compile(checkpointer=memory)

print("🚀 ¡Grafo compilado con éxito y persistencia activada!")

In [ ]:
import os
from IPython.display import Image, Markdown, display

print("\n--- Generando el flujo visual del Grafo vía Mermaid ---")

try:
    # Intentamos renderizar directamente el PNG (Requiere pygraphviz o playwright según tu entorno)
    image_data = graph.get_graph().draw_mermaid_png()
    display(Image(data=image_data))
    print("🎉 ¡Grafo renderizado y exhibido con éxito en el notebook!")

except Exception as e:
    print(f"\nAviso: No se pudo generar el PNG automáticamente. Detalle: {e}")
    print("💡 Tip: Esto suele ocurrir si faltan los binarios de Playwright en el entorno virtual.")
    print("   Podés solucionarlo ejecutando: !pip install playwright && playwright install")
    
    print("\n--- Activando Fallback: Generando código Mermaid en Texto Plano ---")
    try:
        # Si falla el PNG, extraemos el código Mermaid puro
        mermaid_code = graph.get_graph().draw_mermaid()
        print("\n--- Código Mermaid (Copia y pegalo en https://mermaid.live para visualizarlo) ---\n")
        print(mermaid_code)
        
    except Exception as e_mermaid:
        print(f"❌ Error crítico al acceder a la estructura del grafo: {e_mermaid}")

In [ ]:
# 1. Definimos la configuración del hilo usando la convención de LangGraph
config = {"configurable": {"thread_id": "clase06-hilo-alura"}}

# 2. Preparamos el estado inicial del agente
inputs = {
    "task": "Cuál es la diferencia entre LangChain y LangSmith",
    "max_revisions": 2,
    "revision_number": 1,
    "content": []
}

# 3. Ejecutamos el streaming para ver el comportamiento del nuevo Grafo
print("🚀 Iniciando la orquestación de agentes (Estructura Alura)...\n")

for event in graph.stream(inputs, config=config):
    for node_name, output in event.items():
        print(f"📦 Nodo activo finalizado: [{node_name}]")
        
        # Muestra datos útiles si el nodo actualizó el texto o las revisiones
        if "revision_number" in output:
            print(f"   • Número de revisión en el estado: {output['revision_number']}")
            
    print("-" * 60) # Separador visual por cada paso del grafo

In [ ]:
# 1. Cambiamos el thread_id para forzar una ejecución limpia y nueva
config = {"configurable": {"thread_id": "clase06-hilo-NUEVO-01"}}

inputs = {
    "task": "Cuál es la diferencia entre LangChain y LangSmith",
    "max_revisions": 2,
    "revision_number": 1,
    "content": []
}

print("🚀 Iniciando la orquestación... ¡Ahora vas a ver todo el contenido!\n")

for event in graph.stream(inputs, config=config):
    for node_name, output in event.items():
        print(f"📦 【 NODO FINALIZADO: {node_name.upper()} 】")
        
        # --- NUEVO: Imprimimos todo el diccionario que devuelve el nodo ---
        # Esto te va a mostrar las respuestas de Llama, los planes de Tavily, etc.
        for key, value in output.items():
            print(f"🔹 {key}: {value}")
            
    print("\n" + "="*70 + "\n") # Un separador más grande para leer mejor